# Feature extraction: all colour + texture features (no shape)

This notebook extracts **every** feature once and saves it to `features/<split>.csv`. The models (notebooks 03 and 07) then choose which **groups** of columns to use, so nothing has to be extracted again.

* Input: `mini_dataset_preprocessed/` (notebook 05: borders → Shades of Gray → hair removal).
* Rows keep the **same order as `<split>_labels.csv`**, with the columns `relpath`, `label` and `y` (nevus = 0, others = 1).
* Columns are named `<group>__<feature>`. For example, `hist_hsv__hsv_H_bin3` belongs to the group `hist_hsv`.
* It runs in parallel on **all CPU cores** (`N_JOBS`). Progress is saved after every chunk, so if it stops it **resumes** where it left off.

| family | groups |
|---|---|
| colour | `hist_rgb`, `hist_hsv`, `hist_lab` (3×BINS each), `hist3d_hsv`, `hist3d_lab` (BINS3D³), `moments_rgb/hsv/lab` (12 each), `derm_colors` (8), `variegation` (2), `color_clusters` (K+3) |
| texture | `glcm` (21), `lbp` (54), `gabor` (27), `hog_pooled` (12), `hog` (324) |
| with mask (later) | the same groups inside the lesion (`les_*`), plus `skin_stats` and `contrast` |

In [1]:
import os, sys, time, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name in ("Binary", "Multiclass", "notebooks") else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
SEED = 42

from joblib import Parallel, delayed
from feature_extraction import GROUPS, extract_all_from_path, feature_groups

## 0. Configuration

In [2]:
DATA_DIR = PROJECT_ROOT / "mini_dataset_preprocessed"     # output of notebook 05
OUT_DIR = PROJECT_ROOT / "features"
OUT_DIR.mkdir(exist_ok=True)
SPLITS = ["train", "val"]

BINS = 32          # bins per channel of the hist_* groups
BINS3D = 8         # bins per channel of the hist3d_* groups (8 -> 512 features)
K_COLORS = 5       # k of k-means for color_clusters

N_JOBS = os.cpu_count()     # all available cores (each one processes whole images)
CHUNK = 200                 # images per chunk (progress is saved after each chunk)
OVERWRITE = False           # True = extract everything again (e.g. after changing BINS)

# Lesion mask: None for now (no segmentation). Later, e.g.:
#   from preprocessing import segment_lesion
#   def MASK_FN(img): return segment_lesion(img)[0]
MASK_FN = None

TAG = "" if MASK_FN is None else "_mask"
print("cores:", N_JOBS, "| data:", DATA_DIR.name, "| output:", OUT_DIR)

cores: 8 | data: mini_dataset_preprocessed | output: C:\Users\Usuario\PycharmProjects\Melanoma-Challenge-ML-UdG\features


## 1. Image list (same order as the labels CSV)

In [3]:
def list_split(split):
    csv = DATA_DIR / f"{split}_labels.csv"
    if csv.exists():                                   # keep the order of the labels file
        df = pd.read_csv(csv)
        df["relpath"] = df["file"].str.replace("\\", "/", regex=False)
        df = df[["relpath", "label"]]
    else:                                              # fallback: folder listing
        rows = [{"relpath": f.relative_to(DATA_DIR).as_posix(), "label": f.parent.name}
                for f in sorted((DATA_DIR / split).rglob("*.jpg"))]
        df = pd.DataFrame(rows)
    df["y"] = (df["label"].str.lower() != "nevus").astype(int)
    missing = [r for r in df["relpath"] if not (DATA_DIR / r).exists()]
    assert not missing, f"{len(missing)} images not found, e.g. {missing[:3]}"
    return df

lists = {s: list_split(s) for s in SPLITS}
pd.DataFrame({s: d["label"].value_counts() for s, d in lists.items()})

,train,val
label,,
nevus,1017,509
others,983,491


## 2. Extraction (parallel, resumable)

In [4]:
def run_split(split, df):
    out = OUT_DIR / f"{split}{TAG}.csv"
    part = OUT_DIR / f"{split}{TAG}.partial.csv"
    if out.exists() and not OVERWRITE:
        print(f"[exists] {out.name} (OVERWRITE=True to redo)")
        return pd.read_csv(out)
    done = pd.read_csv(part) if (part.exists() and not OVERWRITE) else pd.DataFrame(columns=["relpath"])
    already = set(done["relpath"])
    todo = [r for r in df["relpath"] if r not in already]
    t0 = time.time()
    with Parallel(n_jobs=N_JOBS, batch_size=4) as parallel:
        for i in range(0, len(todo), CHUNK):
            chunk = todo[i:i + CHUNK]
            rows = parallel(delayed(extract_all_from_path)(DATA_DIR / r, MASK_FN, bins=BINS,
                                                           bins3d=BINS3D, k_colors=K_COLORS) for r in chunk)
            done = pd.concat([done, pd.DataFrame(rows).assign(relpath=chunk)], ignore_index=True)
            done.to_csv(part, index=False)
            n = len(done)
            print(f"  {split}: {n}/{len(df)}  ({time.time() - t0:.0f} s)", end="\r")
    feats = df.merge(done, on="relpath", how="left")              # labels order
    assert len(feats) == len(df) and not feats.drop(columns=["relpath", "label", "y"]).isna().all(axis=1).any()
    feats.to_csv(out, index=False)
    part.unlink(missing_ok=True)
    print(f"\n[saved] {out.name}: {len(feats)} images x {feats.shape[1] - 3} features in {time.time() - t0:.0f} s")
    return feats

tables = {s: run_split(s, d) for s, d in lists.items()}

  train: 2000/2000  (293 s)
[saved] train.csv: 2000 images x 1804 features in 296 s
  val: 1000/1000  (130 s)
[saved] val.csv: 1000 images x 1804 features in 132 s


## 3. Summary of the table

In [5]:
F = tables["train"]
summary = pd.DataFrame([{"group": g, "n_features": sum(c.split("__", 1)[0] == g for c in F.columns),
                         "description": GROUPS.get(g.removeprefix("les_"), "")}
                        for g in feature_groups(F.columns)])
print("total features:", summary["n_features"].sum())
summary

total features: 1804


,group,n_features,description
0,color_clusters,8,"k-means: number of distinct colours, cluster s..."
1,derm_colors,8,6 dermoscopic colours + number of colours (8)
2,gabor,27,"Gabor bank, 3 wavelengths x 4 orientations, me..."
3,glcm,21,"GLCM / Haralick, 7 properties x 3 distances (21)"
4,hist3d_hsv,512,joint 3-D histogram HSV (bins3d^3)
5,hist3d_lab,512,joint 3-D histogram Lab (bins3d^3)
6,hist_hsv,96,per-channel histogram HSV (3 x bins)
7,hist_lab,96,per-channel histogram Lab (3 x bins)
8,hist_rgb,96,per-channel histogram RGB (3 x bins)
9,hog,324,"classic HOG on 64x64 (324, whole image only)"
